In [0]:
%run ./01_config

In [0]:
"""
05_silver_conform.py  —  Silver layer: typing and conforming

Part of the SAP PM cost-aware maintenance optimisation pipeline.
Run order is filename order: 01 through 12.
"""

# 05 — Silver conform

# Types the bronze strings, renames SAP technical fields to the vocabulary used in the
# report, derives the EN 13306 policy category from the SAP order type, and folds COSS
# settlement totals onto the order so silver matches the minimum data contract of Section 2.4
# (which specifies total_actual_cost as an order-level field).

# Optional columns (QMNUM_ORIG, CRITICALITY, frailty) are picked up if the extract
# carries them and materialised as NULL if not, so the same code serves v2 and v2.1.

# Shared configuration from '01_config' is assumed to be in scope.

from pyspark.sql import functions as F

use_project_schema()

def col_or_null(df, name, dtype="string"):
    """Return the column if present in the extract, otherwise a typed NULL."""
    return F.col(name).cast(dtype) if name in df.columns else F.lit(None).cast(dtype)

def save(df, name):
    (df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl(name)))
    print(f"{name:34s} {spark.table(tbl(name)).count():>8,} rows")

# Functional locations (IFLOT)

b = spark.table(tbl("bronze_iflot"))

silver_iflot = b.select(
    F.col("TPLNR").alias("functional_location_id"),
    F.col("PLTXT").alias("location_description"),
    F.col("FLTYP").alias("location_type"),
    F.col("WERKS").alias("plant"),
    F.col("IWERK").alias("planning_plant"),
    col_or_null(b, "CRITICALITY", "int").alias("criticality"),
    F.col("_dataset_version"),
).dropDuplicates(["functional_location_id"])

save(silver_iflot, "silver_functional_location")

if "CRITICALITY" not in b.columns:
    print("NOTE: IFLOT carries no CRITICALITY column. Section 2.4 lists it as a required contract "
          "field and Section 4.1.1 says downtime valuation is criticality-weighted. Add it to the "
          "v2.1 generator or amend the contract table in the report.")

# Equipment master (EQUI)

b = spark.table(tbl("bronze_equi"))

silver_equi = b.select(
    F.col("EQUNR").alias("equipment_id"),
    F.col("EQTYP").alias("equipment_class"),
    F.col("TPLNR").alias("functional_location_id"),
    F.col("IWERK").alias("planning_plant"),
    F.col("HERST").alias("manufacturer"),
    F.to_date("INSTALL_DATE").alias("start_up_date"),
    F.col("EQKTX").alias("equipment_description"),
    F.split(F.col("EQTYP"), "-").getItem(0).alias("iso14224_group"),
    col_or_null(b, "FRAILTY", "double").alias("_frailty_disclosed"),
    F.col("_dataset_version"),
).dropDuplicates(["equipment_id"])

save(silver_equi, "silver_equipment")

# Maintenance plans (MPLA) — the incumbent calendar policy

b = spark.table(tbl("bronze_mpla"))

silver_mpla = b.select(
    F.col("WARPL").alias("plan_id"),
    F.col("EQUNR").alias("equipment_id"),
    F.col("WARPL_ART").alias("plan_type"),
    F.col("CYCLE_DAYS").cast("int").alias("cycle_length"),
    F.lit("DAY").alias("cycle_unit"),
    F.to_date("PLAN_START_DATE").alias("plan_start_date"),
    F.col("_dataset_version"),
).dropDuplicates(["plan_id"])

save(silver_mpla, "silver_maintenance_plan")

# Notifications (QMEL)

# M1 = malfunction report (breakdown), M2 = defect report. The escalation linkage
# originating_notification_id is the ground-truth label for the prioritizer (Section 4.3.2).

b = spark.table(tbl("bronze_qmel"))

silver_qmel = b.select(
    F.col("QMNUM").alias("notification_id"),
    F.col("EQUNR").alias("equipment_id"),
    F.col("QMART").alias("notification_type"),
    F.to_date("QMDAT").alias("notification_date"),
    F.to_date("AUSVN").alias("malfunction_start"),
    F.col("PRIOK").cast("int").alias("priority"),
    F.col("FECOD").alias("damage_code"),
    F.col("URCOD").alias("cause_code"),
    F.coalesce(
        col_or_null(b, "QMNUM_ORIG"),
        col_or_null(b, "ORIGINATING_NOTIFICATION_ID"),
    ).alias("originating_notification_id"),
    F.col("_dataset_version"),
).dropDuplicates(["notification_id"])

save(silver_qmel, "silver_notification")

display(silver_qmel.groupBy("notification_type").count().orderBy("notification_type"))

has_escalation = silver_qmel.filter(F.col("originating_notification_id").isNotNull()).limit(1).count() > 0
if not has_escalation:
    print("NOTE: no escalation linkage present. The defect->breakdown mechanism described in "
          "Section 4.1.1 is not in this extract, so FR4 / RQ3 cannot be evaluated on it. Expected "
          "for v2; must be present in v2.1.")

# Orders (AUFK) with settled cost (COSS)

# EN 13306 category derived from the SAP order type rather than from the generator's
# ORDER_CLASS string, so the mapping is auditable against the standard (Test T7).

b = spark.table(tbl("bronze_aufk"))
c = spark.table(tbl("bronze_coss"))

order_type_map = F.create_map(*[F.lit(x) for kv in EN13306_ORDER_TYPES.items() for x in kv])

cost = c.select(
    F.col("AUFNR").alias("order_id"),
    F.col("TOTAL_ACTUAL_COST").cast("double").alias("total_actual_cost"),
    F.col("KSTAR_LABOR").cast("double").alias("cost_labour"),
    F.col("KSTAR_MATERIAL").cast("double").alias("cost_material"),
    F.col("KSTAR_OVERHEAD").cast("double").alias("cost_overhead"),
    # Downtime valuation sits outside the settled order cost, as in SAP. The decision layer
    # adds it explicitly; the cost-ratio calibration check in notebook 07 does not.
    F.coalesce(col_or_null(c, "DOWNTIME_VALUATION", "double"), F.lit(0.0)).alias("downtime_valuation"),
    F.col("COST_TYPE").alias("cost_type"),
).dropDuplicates(["order_id"])

silver_aufk = (
    b.select(
        F.col("AUFNR").alias("order_id"),
        F.col("AUART").alias("order_type"),
        F.col("EQUNR").alias("equipment_id"),
        F.col("QMNUM").alias("notification_id"),
        F.col("WARPL").alias("plan_id"),
        F.col("WERKS").alias("plant"),
        F.to_date("GSTRP").alias("basic_start"),
        F.to_date("IDAT2").alias("basic_finish"),
        F.col("PRIOK").cast("int").alias("priority"),
        F.col("_dataset_version"),
    )
    .dropDuplicates(["order_id"])
    .withColumn("en13306_category", order_type_map[F.col("order_type")])
    .withColumn("order_class",
                F.when(F.col("en13306_category") == "preventive", F.lit("PREVENTIVE"))
                 .otherwise(F.lit("CORRECTIVE")))
    .join(cost, "order_id", "left")
)

save(silver_aufk, "silver_order")

display(silver_aufk.groupBy("order_type", "en13306_category", "order_class")
                   .agg(F.count("*").alias("orders"),
                        F.round(F.avg("total_actual_cost"), 2).alias("avg_cost"))
                   .orderBy("order_type"))

# Confirmations (AFRU)

b = spark.table(tbl("bronze_afru"))

silver_afru = b.select(
    F.col("RUECK").alias("confirmation_id"),
    F.col("AUFNR").alias("order_id"),
    F.to_date("BUDAT").alias("posting_date"),
    F.col("ISMNW").cast("double").alias("actual_work_hours"),
    F.col("ARBPL").alias("work_centre"),
    col_or_null(b, "PERSONNEL").alias("technician_token"),
    F.col("_dataset_version"),
).dropDuplicates(["confirmation_id"])

save(silver_afru, "silver_confirmation")

# Referential integrity

# Fails loudly rather than silently producing orphan rows that would distort the
# survival intervals built in notebook 06.

checks = [
    ("silver_order.equipment_id -> silver_equipment", "silver_order", "equipment_id", "silver_equipment", "equipment_id"),
    ("silver_notification.equipment_id -> silver_equipment", "silver_notification", "equipment_id", "silver_equipment", "equipment_id"),
    ("silver_maintenance_plan.equipment_id -> silver_equipment", "silver_maintenance_plan", "equipment_id", "silver_equipment", "equipment_id"),
    ("silver_confirmation.order_id -> silver_order", "silver_confirmation", "order_id", "silver_order", "order_id"),
    ("silver_equipment.functional_location_id -> silver_functional_location", "silver_equipment", "functional_location_id", "silver_functional_location", "functional_location_id"),
]

problems = []
for label, child, ck, parent, pk in checks:
    orphans = (spark.table(tbl(child)).filter(F.col(ck).isNotNull())
                    .join(spark.table(tbl(parent)).select(F.col(pk).alias("_pk")),
                          F.col(ck) == F.col("_pk"), "left_anti")
                    .count())
    status = "PASS" if orphans == 0 else "FAIL"
    print(f"{status}  {label}: {orphans} orphan(s)")
    if orphans:
        problems.append(label)

if problems:
    raise ValueError(f"Referential integrity failures: {problems}")
print("\nAll referential integrity checks passed.")